In [1]:
from tempfile import TemporaryDirectory

# imports
import numpy as np
import numpy.linalg as LA

from base.qda import QDA, TensorizedQDA, FasterQDA
from base.cholesky import QDA_Chol1, QDA_Chol2, QDA_Chol3
from utils.datasets import (get_iris_dataset, get_letters_dataset,
                            get_penguins_dataset, get_wine_dataset,
                            label_encode)
from utils.datasets import split_transpose

In [2]:
# levantamos el dataset Wine, que tiene 13 features y 178 observaciones en total
X_full, y_full = get_wine_dataset()
y_full_encoded = label_encode(y_full)

In [3]:
FQDA = FasterQDA()

test_sz = 0.3
rng = 6663

# Generate current train-test split
X_train, X_test, y_train, y_test = split_transpose(
    X_full, y_full_encoded,
    test_size=test_sz,
    random_state=rng
)
FQDA.fit(X_train, y_train)

In [4]:
FQDA.predict(X_test)

array([[0, 0, 1, 1, 1, 2, 0, 0, 2, 1, 2, 0, 0, 2, 2, 1, 0, 0, 1, 0, 0, 0,
        1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 2, 1, 1, 1, 0, 2, 0, 1, 1, 0, 0,
        0, 1, 0, 0, 0, 1, 1, 0, 1, 0]])

## 4. ¿Dónde aparece la matriz de `n × n`?

En `FasterQDA.predict`, la línea que calcula `cross_products` es la que construye esa matriz:

```python
cross_products = centered_X.transpose(0, 2, 1) @ self.tensor_inv_cov @ centered_X
```

Si hay `k` clases, `p` features y `n` observaciones, `centered_X` tiene shape `(k, p, n)` y `self.tensor_inv_cov` tiene `(k, p, p)`. El producto tiene entonces shape `(k, n, n)`: **para cada clase se genera una matriz de `n × n`** con las interacciones entre observaciones. En este ejemplo, con 3 clases y 54 observaciones de prueba, `cross_products` tiene shape `(3, 54, 54)`.

`np.diagonal(cross_products, axis1=-2, axis2=-1)` extrae los `n` términos necesarios para cada clase y devuelve un array de shape `(k, n)`, pero la matriz completa ya fue calculada.